# Week 3 — CoNLL-2003 Dataset Acquisition, Validation & EDA

**Project:** Healthcare Form Entity Extractor  
**Dataset:** CoNLL-2003  
**Model Plan:** BiLSTM-CRF  
**Week:** 3 — Data Acquisition, Validation & Exploratory Data Analysis (EDA)

## Objective

This notebook validates the CoNLL-2003 dataset and performs exploratory data analysis before model development.

We will analyze:

- Dataset structure and files
- Train, development and test splits
- Number of sentences and tokens
- Missing and malformed records
- Duplicate sentences
- NER labels and label distribution
- Sentence lengths
- Token frequencies
- Differences between train, development and test sets

In [2]:
import os 

from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
DATA_DIR = "../../data/raw/conll2003/"

train_file = os.path.join(DATA_DIR, "eng.train")
dev_file = os.path.join(DATA_DIR, "eng.testa")
test_file = os.path.join(DATA_DIR, "eng.testb")

print("Dataset files:")
print("Train:", train_file)
print("Development:", dev_file)
print("Test:", test_file)

Dataset files:
Train: ../../data/raw/conll2003/eng.train
Development: ../../data/raw/conll2003/eng.testa
Test: ../../data/raw/conll2003/eng.testb


In [4]:
files = {
    "Train" : train_file,
    "Development" : dev_file,
    "Test" : test_file
}

for name, path in files.items():
    print(f"{name}:{'FOUND' if os.path.exists(path) else 'NOT FOUND'}")

Train:FOUND
Development:FOUND
Test:FOUND


In [5]:
# Validate the structure of the CoNLL-2003 dataset

def validate_conll_file(file_path):
    total_rows = 0
    valid_rows = 0
    malformed_rows = 0
    empty_rows = 0

    with open(file_path, "r", encoding = "utf-8") as file:
        for line_number, line in enumerate(file, start = 1):
            line = line.rstrip("\n")

            # Empty line = sentence separator
            if not line.rstrip():
                empty_rows += 1
                continue;

            # Ignore document start markers
            if line.startswith("-DOCSTART-"):
                continue;

            total_rows += 1

            parts = line.split()

            # Every data row should have 4 columns
            if len(parts) == 4:
                valid_rows += 1
            else:
                malformed_rows += 1

                print(
                    f"Malformed row in {file_path} "
                    f"at line {line_number}: {line}"
                )
    return {
        "Total_data_rows" : total_rows,
        "Valid_row" : valid_rows,
        "Malformed" : malformed_rows,
        "Empty_rows" : empty_rows
    }

# Validate all dataset splits

validation_results = {}
for name, path in files.items():
    validation_results[name] = validate_conll_file(path)

validation_df = pd.DataFrame(validation_results).T

validation_df

,Total_data_rows,Valid_row,Malformed,Empty_rows
Train,203621,203621,0,14987
Development,51362,51362,0,3466
Test,46435,46435,0,3684


In [6]:
# Load CoNLL-2003 data into sentences

def load_conll_file(file_path):
    sentences = []
    current_sentence = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            # Empty line means the sentence has ended
            if not line:
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
                continue

            # Ignore document start markers
            if line.startswith("-DOCSTART-"):
                continue

            parts = line.split()

            # Expected format: Token, POS, Chunk, NER
            if len(parts) == 4:
                token, pos, chunk, ner_label = parts

                current_sentence.append({
                    "token": token,
                    "pos": pos,
                    "chunk": chunk,
                    "ner": ner_label
                })

        # Add the final sentence if needed
        if current_sentence:
            sentences.append(current_sentence)

    return sentences


# Load all three splits
train_data = load_conll_file(train_file)
dev_data = load_conll_file(dev_file)
test_data = load_conll_file(test_file)

print("Dataset loaded successfully!")
print("Train sentences:", len(train_data))
print("Development sentences:", len(dev_data))
print("Test sentences:", len(test_data))


Dataset loaded successfully!
Train sentences: 14041
Development sentences: 3250
Test sentences: 3453


In [7]:
# Check for missing or invalid NER labels

valid_labels = {
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
}


def check_ner_labels(file_path):
    missing_labels = 0
    invalid_labels = Counter()

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line or line.startswith("-DOCSTART-"):
                continue

            parts = line.split()

            if len(parts) == 4:
                ner_label = parts[3]

                if not ner_label:
                    missing_labels += 1
                elif ner_label not in valid_labels:
                    invalid_labels[ner_label] += 1

    return missing_labels, invalid_labels


label_results = {}

for name, path in files.items():
    missing, invalid = check_ner_labels(path)

    label_results[name] = {
        "Missing labels": missing,
        "Invalid labels": dict(invalid)
    }

label_results

{'Train': {'Missing labels': 0, 'Invalid labels': {}},
 'Development': {'Missing labels': 0, 'Invalid labels': {}},
 'Test': {'Missing labels': 0, 'Invalid labels': {}}}

In [8]:
# Check for duplicate sentences

def find_duplicate_sentences(data):
    sentence_counts = Counter()

    for sentence in data:
        tokens = tuple(item["token"] for item in sentence)
        sentence_counts[tokens] += 1

    duplicate_sentences = {
        sentence: count
        for sentence, count in sentence_counts.items()
        if count > 1
    }

    return duplicate_sentences


duplicate_results = {}

for name, data in {
    "Train": train_data,
    "Development": dev_data,
    "Test": test_data
}.items():

    duplicates = find_duplicate_sentences(data)

    duplicate_results[name] = {
        "Unique duplicate sentences": len(duplicates),
        "Total repeated occurrences": sum(duplicates.values())
    }

duplicate_df = pd.DataFrame(duplicate_results).T

duplicate_df

,Unique duplicate sentences,Total repeated occurrences
Train,721,2071
Development,95,275
Test,147,416
